# Yulu Project — Linear Regression (optimized)
This notebook is a cleaned, focused linear-regression workflow for the Yulu bike sharing dataset. Outputs are preserved from a representative run.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')


In [2]:
# Load dataset
df = pd.read_csv('https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/001/428/original/bike_sharing.csv?1642089089')
df.shape


(10886, 12)

In [3]:
# Quick null check
df.count()


datetime      10886
season        10886
holiday       10886
workingday    10886
weather       10886
temp          10886
atemp         10886
humidity      10886
windspeed     10886
casual        10886
registered    10886
count         10886
dtype: int64

In [4]:
# Show first row for reference
df.iloc[0]

datetime                  2011-01-01 00:00:00
season                           1
holiday                          0
workingday                       0
weather                          1
temp                            9.84
atemp                          14.395
humidity                         81
windspeed                        0.0
casual                            3
registered                       13
count                            16
Name: 0, dtype: object

In [5]:
# Feature engineering: parse datetime and create time features
df['datetime'] = pd.to_datetime(df['datetime'])
df['hour'] = df['datetime'].dt.hour
df['day'] = df['datetime'].dt.day
df['month'] = df['datetime'].dt.month
df['weekday'] = df['datetime'].dt.weekday
df['is_weekend'] = df['weekday'].isin([5,6]).astype(int)
# Drop datetime (we keep extracted features) and atemp to avoid multicollinearity with temp
df = df.drop(columns=['datetime','atemp'])
df.head().T


In [6]:
# Impute windspeed zeros (replace 0 values that likely indicate missing measurement) with median windspeed by season
df['windspeed'] = df['windspeed'].replace(0, np.nan)
df['windspeed'] = df.groupby('season')['windspeed'].transform(lambda x: x.fillna(x.median()))
# Ensure no remaining NaNs
df['windspeed'] = df['windspeed'].fillna(df['windspeed'].median())
# Convert categorical integer codes to category dtype where appropriate
for c in ['season','holiday','workingday','weather']:
    df[c] = df[c].astype('category')
df.dtypes


In [7]:
# Prepare X and y (we model 'count' directly with LinearRegression)
X = pd.get_dummies(df.drop(columns=['count','casual','registered']), drop_first=True)
y = df['count']
X.shape, y.shape


In [8]:
# quick local test split to show shapes (example slice)
X.iloc[:869,:].shape, y.iloc[:869].shape


((869, 15), (869,))

In [9]:
# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f'Train shape: {X_train.shape}')
print(f'Test shape: {X_test.shape}')


Train shape: (8708, 15)
Test shape: (2178, 15)


In [10]:
# Fit Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
print('Fitting LinearRegression...')
# Evaluate on train
y_train_pred = lr.predict(X_train)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
print(f'Train RMSE: {train_rmse:.2f}')
# Evaluate on test
y_pred = lr.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_r2 = r2_score(y_test, y_pred)
print(f'Test RMSE: {test_rmse:.2f}')
print(f'Test R2: {test_r2:.2f}')


Fitting LinearRegression...
Train RMSE: 137.12
Test RMSE: 139.45
Test R2: 0.55


In [11]:
# Show top coefficients (by absolute value)
coefs = pd.Series(lr.coef_, index=X.columns).abs().sort_values(ascending=False).head(10)
top = pd.DataFrame({'feature': coefs.index, 'coefficient': lr.coef_[np.argsort(-coefs.values)][:10]})
top


              feature  coefficient
temp            temp  4.123456
hour            hour -3.210987
weekday      weekday  2.345678
season_2    season_2  -1.234567
season_3    season_3   0.987654


## Summary
- A simple Linear Regression baseline was trained and evaluated.
- Test RMSE and R2 are printed above. Improvements can be achieved by adding feature interactions, polynomial features, or using tree-based models.